# Bird Aero — Full Training (Colab GPU)

Train MeshGraphNet on the DeepMind deforming plate dataset with proper hyperparameters.
Run this on Colab with a T4/A100 GPU. Download the checkpoint when done.

**Expected time on T4**: ~20-30 min for 100 trajectories, 20 epochs

In [ ]:
# 1. Install
!git clone https://github.com/orireshef/birdaero-vibration-poc.git 2>/dev/null || echo 'Already cloned'
%cd birdaero-vibration-poc
!pip install -e . -q

import sys
sys.path.insert(0, 'src')

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Config

Adjust these parameters based on available GPU memory and time budget.

In [ ]:
# ── Adjust these ──
MAX_TRAJECTORIES = 100   # of 1000 available. 100 = ~40K graphs. Use None for all.
EPOCHS = 20              # 20-50 for good results
HIDDEN_DIM = 64          # 64 = production size
NUM_LAYERS = 8           # 8 = production size (15 in original paper)
LEARNING_RATE = 1e-4
LR_DECAY = 0.9999991
BATCH_SIZE = 1           # GNN graphs have variable size, keep at 1

# Physics-aware training (set weights > 0 to enable)
BC_LOSS_WEIGHT = 1.0     # boundary condition enforcement
STRESS_LOSS_WEIGHT = 0.0 # stress prediction (set 0.1 to enable)
SMOOTHNESS_WEIGHT = 0.01 # displacement smoothness

print(f'Training config:')
print(f'  Trajectories: {MAX_TRAJECTORIES}')
print(f'  Epochs: {EPOCHS}')
print(f'  Model: hidden={HIDDEN_DIM}, layers={NUM_LAYERS}')
print(f'  Physics: BC={BC_LOSS_WEIGHT}, stress={STRESS_LOSS_WEIGHT}, smooth={SMOOTHNESS_WEIGHT}')
print(f'  Estimated graphs: ~{MAX_TRAJECTORIES * 399 if MAX_TRAJECTORIES else "400K"}')

## 3. Download Dataset

In [ ]:
%%time
from vibration_poc.dataset.config import DatasetConfig
from vibration_poc.dataset.download import download_dataset

dataset_config = DatasetConfig(max_trajectories=MAX_TRAJECTORIES)
download_dataset(dataset_config)
!ls -lh {dataset_config.raw_dir}

## 4. Preprocess

Parse TFRecords → graph dicts (.pt files). Streams to disk to avoid OOM.
Edge topology computed once per trajectory (mesh is static across frames).

In [ ]:
%%time
from vibration_poc.dataset.preprocess import preprocess_dataset

preprocess_dataset(dataset_config)

from pathlib import Path
for split in dataset_config.splits:
    n = len(list((dataset_config.processed_dir / split).glob('*.pt')))
    print(f'  {split}: {n:,} graphs')

## 5. Train

In [ ]:
%%time
from vibration_poc.training.trainer import TrainingConfig, train
from vibration_poc.physics import PhysicsConfig

predict_stress = STRESS_LOSS_WEIGHT > 0

training_config = TrainingConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_decay=LR_DECAY,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    batch_size=BATCH_SIZE,
    device=device,
    checkpoint_dir='checkpoints',
    log_interval=1,
    physics=PhysicsConfig(
        bc_node_types=[1, 2, 3],
        bc_loss_weight=BC_LOSS_WEIGHT,
        stress_loss_weight=STRESS_LOSS_WEIGHT,
        smoothness_weight=SMOOTHNESS_WEIGHT,
    ),
)

best_path = train(training_config, dataset_config)
print(f'\nBest checkpoint: {best_path}')

## 6. Quick Validation

In [ ]:
import json
import numpy as np
from vibration_poc.engine import load_model, evaluate_design
from vibration_poc.dataset.config import NormStats
from vibration_poc.dataset.preprocess import load_meta, _iter_graphs

# Load model
model = load_model(
    best_path,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    predict_stress=predict_stress,
    device=device,
)

# Load norm stats
with open(dataset_config.processed_dir / 'norm_stats.json') as f:
    norm_stats = NormStats(**json.load(f))

# Get a raw (unnormalized) test graph for rollout
meta = load_meta(dataset_config.raw_dir / 'meta.json')
test_graph = next(_iter_graphs(
    dataset_config.raw_dir / 'test.tfrecord', meta, max_trajectories=1
))

# Run evaluation
metrics, results = evaluate_design(
    test_graph, model, norm_stats,
    num_steps=50,
    bc_node_types=[1, 2, 3],
)

# Compare with ground truth
gt_disp = test_graph['y'].numpy()
gt_mag = np.linalg.norm(gt_disp, axis=1)

print(f'Ground truth:  max={gt_mag.max()*1000:.3f} mm, mean={gt_mag.mean()*1000:.3f} mm')
print(f'Prediction:    max={metrics.max_displacement*1000:.3f} mm, mean={metrics.mean_displacement*1000:.3f} mm')
print(f'Pred/GT ratio: {metrics.max_displacement / max(gt_mag.max(), 1e-10):.2f}x')
print(f'\nDominant frequencies:')
for freq, mag in metrics.dominant_frequencies[:3]:
    print(f'  {freq:.2f} Hz — magnitude {mag:.4f}')

## 7. Download Checkpoint

Download the trained checkpoint + norm stats to use locally with the Streamlit demo.

In [ ]:
import shutil

# Bundle checkpoint + norm stats + config for local use
export_dir = Path('export')
export_dir.mkdir(exist_ok=True)

shutil.copy(best_path, export_dir / 'best.pt')
shutil.copy(
    dataset_config.processed_dir / 'norm_stats.json',
    export_dir / 'norm_stats.json',
)

# Save model config so we know how to load it
config_info = {
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'predict_stress': predict_stress,
    'epochs': EPOCHS,
    'max_trajectories': MAX_TRAJECTORIES,
}
with open(export_dir / 'model_config.json', 'w') as f:
    json.dump(config_info, f, indent=2)

# Save a raw test graph for demo
import torch
torch.save(test_graph, export_dir / 'test_sample.pt')

# Zip for download
shutil.make_archive('birdaero_checkpoint', 'zip', export_dir)
print(f'Export ready: birdaero_checkpoint.zip')
print(f'Contents:')
for f in sorted(export_dir.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1024:.0f} KB)')

In [ ]:
# Download (Colab only)
try:
    from google.colab import files
    files.download('birdaero_checkpoint.zip')
except ImportError:
    print('Not running on Colab. Find birdaero_checkpoint.zip in the working directory.')

## Local Usage

After downloading, unzip and use with the Streamlit demo:

```bash
unzip birdaero_checkpoint.zip -d checkpoint_export/
cp checkpoint_export/best.pt checkpoints/best.pt
cp checkpoint_export/norm_stats.json data/processed/deforming_plate/norm_stats.json

# Launch demo — set hidden_dim and num_layers to match model_config.json
make demo
```

Upload `checkpoint_export/test_sample.pt` in the Streamlit app as your mesh file.